# Preparing Data for Scikit-Learn

## Definition
Scikit-learn expects:
- **X**: 2D array-like, shape `(n_samples, n_features)`, **no NaN**, **numeric only**
- **y**: 1D array-like, shape `(n_samples,)`

### Complete Preprocessing Pipeline
1. Drop ID/irrelevant columns
2. Handle missing values
3. Encode categoricals
4. Scale numeric features
5. Feature/target separation
6. Train/test split
7. Fit preprocessing on train, transform both

### Using sklearn Pipelines (best practice)
```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(), cat_cols)
])
pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
```

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

rng = np.random.default_rng(42)
n = 500

df = pd.DataFrame({
    'age':        rng.integers(18, 65, n),
    'income':     rng.normal(60000, 20000, n).clip(0),
    'score':      rng.normal(700, 80, n).clip(300, 850),
    'employment': rng.choice(['Employed', 'Self-Employed', 'Unemployed'], n),
    'education':  rng.choice(['High School', 'Bachelor', 'Master'], n),
    'default':    rng.choice([0, 1], n, p=[0.85, 0.15])
})
# Inject missing values
df.loc[rng.choice(n, 30, replace=False), 'income'] = np.nan
df.loc[rng.choice(n, 20, replace=False), 'employment'] = np.nan

print('Dataset:', df.shape)
print('Missing:', df.isna().sum().to_dict())

In [ ]:
# Step 1: Feature/target split
X = df.drop(columns=['default'])
y = df['default']

numeric_cols = ['age', 'income', 'score']
cat_cols = ['employment', 'education']

# Step 2: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Step 3: Build ColumnTransformer
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, cat_cols)
])

print('Preprocessor built.')

In [ ]:
# Step 4: Full ML Pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Fit on train, predict on test (no leakage!)
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Inspect preprocessed feature names
preprocessor.fit(X_train)
num_feature_names = numeric_cols
cat_feature_names = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(cat_cols).tolist()
all_features = num_feature_names + cat_feature_names
print('All feature names after preprocessing:')
for f in all_features:
    print(f'  {f}')

In [ ]:
# Getting preprocessed array as DataFrame (for inspection)
X_train_transformed = preprocessor.transform(X_train)
X_train_df = pd.DataFrame(X_train_transformed, columns=all_features)
print('Preprocessed train data:')
print(X_train_df.head(3).round(3))
print('No NaN:', X_train_df.isna().sum().sum() == 0)